> **Weather-aware workflow note:** The current screening expected-power series is still Persistence. Weather-aware ML forecasts are evaluated separately and do not alter this screening rule.

In [1]:
# What this cell does:
# Import the libraries required for the Digital Twin analysis.

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# What this cell does:
# Define the locations of the final prediction files
# generated during model evaluation.

PREDICTIONS_PATH = "../outputs/predictions/"
METRICS_PATH = "../outputs/metrics/"

print("Prediction files:")
print(os.listdir(PREDICTIONS_PATH))

Prediction files:
['plant1_persistence_predictions.csv', 'plant1_test_predictions.csv', 'plant2_persistence_predictions.csv', 'plant2_test_predictions.csv']


In [3]:
# What this cell does:
# Load the final test predictions for both plants.

plant1_twin = pd.read_csv(
    PREDICTIONS_PATH + "plant1_test_predictions.csv"
)

plant2_twin = pd.read_csv(
    PREDICTIONS_PATH + "plant2_test_predictions.csv"
)

print("Plant 1 shape:", plant1_twin.shape)
print("Plant 2 shape:", plant2_twin.shape)

Plant 1 shape: (10076, 9)
Plant 2 shape: (10428, 9)


In [4]:
# What this cell does:
# Convert DATE_TIME from string to datetime format
# so that the Digital Twin data can be sorted and analyzed chronologically.

plant1_twin["DATE_TIME"] = pd.to_datetime(
    plant1_twin["DATE_TIME"]
)

plant2_twin["DATE_TIME"] = pd.to_datetime(
    plant2_twin["DATE_TIME"]
)

plant1_twin = plant1_twin.sort_values("DATE_TIME").reset_index(drop=True)
plant2_twin = plant2_twin.sort_values("DATE_TIME").reset_index(drop=True)

print("Plant 1 time range:")
print(plant1_twin["DATE_TIME"].min(), "→", plant1_twin["DATE_TIME"].max())

print("\nPlant 2 time range:")
print(plant2_twin["DATE_TIME"].min(), "→", plant2_twin["DATE_TIME"].max())

Plant 1 time range:
2020-06-13 04:30:00 → 2020-06-17 23:30:00

Plant 2 time range:
2020-06-13 01:15:00 → 2020-06-17 23:30:00


In [5]:
# What this cell does:
# Check that the actual target and selected model prediction
# are available before creating the Digital Twin variables.

print("Plant 1 columns:")
print(plant1_twin.columns.tolist())

print("\nPlant 2 columns:")
print(plant2_twin.columns.tolist())

Plant 1 columns:
['DATE_TIME', 'PLANT_ID', 'SOURCE_KEY', 'AC_POWER', 'TARGET_AC_POWER', 'Persistence', 'Linear_Regression', 'Random_Forest', 'XGBoost']

Plant 2 columns:
['DATE_TIME', 'PLANT_ID', 'SOURCE_KEY', 'AC_POWER', 'TARGET_AC_POWER', 'Persistence', 'Linear_Regression', 'Random_Forest', 'XGBoost']


In [6]:
# What this cell does:
# Use the selected final forecasting model to define
# expected AC power for the Digital Twin.

plant1_twin["EXPECTED_AC_POWER"] = plant1_twin["Persistence"]

plant2_twin["EXPECTED_AC_POWER"] = plant2_twin["Persistence"]

print("Expected power created successfully.")

Expected power created successfully.


In [7]:
# What this cell does:
# Rename the 15-minute-ahead target as the actual
# AC power observed at the forecast horizon.

plant1_twin["ACTUAL_AC_POWER"] = plant1_twin["TARGET_AC_POWER"]

plant2_twin["ACTUAL_AC_POWER"] = plant2_twin["TARGET_AC_POWER"]

print("Actual power created successfully.")

Actual power created successfully.


In [8]:
# What this cell does:
# Calculate the difference between actual and expected AC power.
#
# Residual = Actual Power - Expected Power

plant1_twin["RESIDUAL"] = (
    plant1_twin["ACTUAL_AC_POWER"]
    - plant1_twin["EXPECTED_AC_POWER"]
)

plant2_twin["RESIDUAL"] = (
    plant2_twin["ACTUAL_AC_POWER"]
    - plant2_twin["EXPECTED_AC_POWER"]
)

print("Residual calculated successfully.")

Residual calculated successfully.


In [9]:
# What this cell does:
# Calculate percentage deviation between actual
# and expected AC power.

plant1_twin["DEVIATION_PCT"] = np.where(
    plant1_twin["EXPECTED_AC_POWER"] > 0,
    (
        plant1_twin["RESIDUAL"]
        / plant1_twin["EXPECTED_AC_POWER"]
    ) * 100,
    0
)

plant2_twin["DEVIATION_PCT"] = np.where(
    plant2_twin["EXPECTED_AC_POWER"] > 0,
    (
        plant2_twin["RESIDUAL"]
        / plant2_twin["EXPECTED_AC_POWER"]
    ) * 100,
    0
)

print("Deviation percentage calculated successfully.")

Deviation percentage calculated successfully.


In [10]:
# What this cell does:
# Display the core Digital Twin variables.

twin_columns = [
    "DATE_TIME",
    "SOURCE_KEY",
    "ACTUAL_AC_POWER",
    "EXPECTED_AC_POWER",
    "RESIDUAL",
    "DEVIATION_PCT"
]

print("Plant 1 Digital Twin:")
display(plant1_twin[twin_columns].head(10))

print("\nPlant 2 Digital Twin:")
display(plant2_twin[twin_columns].head(10))

Plant 1 Digital Twin:


,DATE_TIME,SOURCE_KEY,ACTUAL_AC_POWER,EXPECTED_AC_POWER,RESIDUAL,DEVIATION_PCT
0,2020-06-13 04:30:00,1BY6WEcLGh8j5v7,0.0,0.0,0.0,0.0
1,2020-06-13 04:30:00,sjndEbLyjtCKgGv,0.0,0.0,0.0,0.0
2,2020-06-13 04:30:00,rGa61gmuvPhdLxV,0.0,0.0,0.0,0.0
3,2020-06-13 04:30:00,zBIq5rxdHJRwDNY,0.0,0.0,0.0,0.0
4,2020-06-13 04:30:00,7JYdWkrLSPkdwr4,0.0,0.0,0.0,0.0
5,2020-06-13 04:30:00,zVJPv84UY57bAof,0.0,0.0,0.0,0.0
6,2020-06-13 04:30:00,ZnxXDlPa8U1GXgE,0.0,0.0,0.0,0.0
7,2020-06-13 04:30:00,z9Y9gH1T5YWrNuG,0.0,0.0,0.0,0.0
8,2020-06-13 04:30:00,WRmjgnKYAwPKWDb,0.0,0.0,0.0,0.0
9,2020-06-13 04:30:00,uHbuxQJl8lW7ozc,0.0,0.0,0.0,0.0



Plant 2 Digital Twin:


,DATE_TIME,SOURCE_KEY,ACTUAL_AC_POWER,EXPECTED_AC_POWER,RESIDUAL,DEVIATION_PCT
0,2020-06-13 01:15:00,LlT2YUhhzqhg5Sw,0.0,0.0,0.0,0.0
1,2020-06-13 01:15:00,rrq4fwE8jgrTyWY,0.0,0.0,0.0,0.0
2,2020-06-13 01:15:00,xMbIugepa2P7lBB,0.0,0.0,0.0,0.0
3,2020-06-13 01:15:00,xoJJ8DcxJEcupym,0.0,0.0,0.0,0.0
4,2020-06-13 01:15:00,WcxssY2VbP4hApt,0.0,0.0,0.0,0.0
5,2020-06-13 01:15:00,81aHJ1q11NBPMrL,0.0,0.0,0.0,0.0
6,2020-06-13 01:15:00,4UPUqMRk7TRMgml,0.0,0.0,0.0,0.0
7,2020-06-13 01:15:00,V94E5Ben1TlhnDV,0.0,0.0,0.0,0.0
8,2020-06-13 01:15:00,IQ2d7wF4YD8zU1Q,0.0,0.0,0.0,0.0
9,2020-06-13 01:15:00,LYwnQax7tkwH5Cb,0.0,0.0,0.0,0.0
